## Import libraries and connect to google drive

In [ ]:
# here we are importing all the required libraries
import tensorflow as tf
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout, RandomFlip, RandomRotation
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50
import matplotlib.pyplot as plt
import os
import numpy as np

# here we are connecting to google drive
print("connecting to google drive...")
from google.colab import drive
drive.mount('/content/drive')
print("drive mounted successfully")


##unzip dataset and define main variables

In [ ]:
# here we are unzipping the dataset from google drive
print("unzipping the data zip file from drive...")
ZIP_PATH = "/content/drive/MyDrive/DATA.zip"
!unzip -q {ZIP_PATH} -d "/content/"
print("data is unzipped and ready in content")

# here we are defining the main project variables
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 4
EPOCHS = 20

# here we are setting the dataset folder paths
TRAIN_DIR = "/content/DATA/Training(70%)"
VALID_DIR = "/content/DATA/Validation(20%)"
TEST_DIR = "/content/DATA/Testing(10%)"


## load and prepare datasets

In [ ]:
# here we are loading the training validation and test datasets
print("loading training validation and test datasets...")
train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    label_mode="categorical",
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    VALID_DIR,
    label_mode="categorical",
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    label_mode="categorical",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# here we are checking the class names
class_names = train_dataset.class_names
print(f"our class names are: {class_names}")

# here we are optimizing data pipelines for better performance
print("optimizing data pipelines with cache and prefetch")
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)
test_dataset = test_dataset.cache().prefetch(buffer_size=AUTOTUNE)


##build the resnet50 baseline model

In [ ]:
# here we are building the baseline resnet50 model
print("building the baseline resnet50 model")
base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3)
)

# here we are freezing the base model for the baseline
print("freezing the resnet50 base layers")
base_model.trainable = False

# here we are defining the input and adding data augmentation
inputs = Input(shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3))
print("adding data augmentation randomflip and randomrotation")
x = RandomFlip('horizontal')(inputs)
x = RandomRotation(0.1)(x)

# here we are passing the augmented data to the frozen base model
x = base_model(x, training=False)

# here we are adding the classification head
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)
model = Model(inputs, outputs)


## compile and train the model

In [ ]:
# here we are compiling the model
print("compiling the model")
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

# here we are showing the model summary
print("model summary")
model.summary()

# here we are starting the baseline training
print("starting model training")
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS,
    verbose=1
)
print("baseline training complete")


## evaluate the baseline model

In [ ]:
# here we are evaluating the baseline model on the test set
print("evaluating the baseline model on the test set")
results = model.evaluate(test_dataset, verbose=1)

# here we are storing the metrics in a dictionary
metrics = {}
metrics['loss'] = results[0]
metrics['accuracy'] = results[1]
metrics['precision'] = results[2]
metrics['recall'] = results[3]

# here we are calculating the f1 score
if (metrics['precision'] + metrics['recall']) > 0:
    metrics['f1_score'] = 2 * (metrics['precision'] * metrics['recall']) / (metrics['precision'] + metrics['recall'])
else:
    metrics['f1_score'] = 0.0

# here we are printing the test results
print("baseline model test results")
print(f"test loss: {metrics['loss']:.4f}")
print(f"test accuracy: {metrics['accuracy']:.4f}")
print(f"test precision: {metrics['precision']:.4f}")
print(f"test recall: {metrics['recall']:.4f}")
print(f"test f1 score: {metrics['f1_score']:.4f}")


## save the trained model

In [ ]:
# here we are saving the trained model to google drive
print("saving the model to google drive")
os.makedirs("/content/drive/MyDrive/MODELS", exist_ok=True)
model.save("/content/drive/MyDrive/MODELS/resnet_baseline.h5")

# here we are confirming that the baseline model is saved
print("baseline resnet model saved to drive")
print("we can now use these results for our report")


Connecting to Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted successfully.
Unzipping the DATA.zip file from Drive...
replace /content/DATA/Testing(10%)/glioma_tumor/image(1).jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: Data is unzipped and ready in /content/.
Loading Training, Validation, and Test datasets...
Found 2297 files belonging to 4 classes.
Found 573 files belonging to 4 classes.
Found 394 files belonging to 4 classes.
Our class names are: ['glioma_tumor', 'meningioma_tumor', 'no_tumor', 'pituitary_tumor']
Optimizing data pipelines with .cache() and .prefetch()...
Building the baseline ResNet50 model...
Freezing the ResNet50 base layers.
Adding Data Augmentation (RandomFlip and RandomRotation)...
Compiling the model (Adam optimizer, categorical crossentropy)...
--- Model Summary ---


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip (RandomFlip)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_rotation                 │ (None, 224, 224, 3)    │             0 │
│ (RandomRotation)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 4)              │         8,196 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,595,908 (90.01 MB)

 Trainable params: 8,196 (32.02 KB)

 Non-trainable params: 23,587,712 (89.98 MB)


--- Starting Model Training ---
Epoch 1/20
72/72 ━━━━━━━━━━━━━━━━━━━━ 30s 236ms/step - accuracy: 0.5095 - loss: 1.2495 - precision: 0.5571 - recall: 0.4086 - val_accuracy: 0.8063 - val_loss: 0.5450 - val_precision: 0.8551 - val_recall: 0.7417
Epoch 2/20
72/72 ━━━━━━━━━━━━━━━━━━━━ 12s 161ms/step - accuracy: 0.7522 - loss: 0.6128 - precision: 0.7981 - recall: 0.7178 - val_accuracy: 0.8290 - val_loss: 0.4640 - val_precision: 0.8673 - val_recall: 0.7871
Epoch 3/20
72/72 ━━━━━━━━━━━━━━━━━━━━ 12s 163ms/step - accuracy: 0.8105 - loss: 0.4945 - precision: 0.8362 - recall: 0.7786 - val_accuracy: 0.8412 - val_loss: 0.4283 - val_precision: 0.8691 - val_recall: 0.7993
Epoch 4/20
72/72 ━━━━━━━━━━━━━━━━━━━━ 12s 165ms/step - accuracy: 0.8161 - loss: 0.4632 - precision: 0.8446 - recall: 0.7856 - val_accuracy: 0.8290 - val_loss: 0.4558 - val_precision: 0.8606 - val_recall: 0.7976
Epoch 5/20
72/72 ━━━━━━━━━━━━━━━━━━━━ 12s 166ms/step - accuracy: 0.8317 - loss: 0.4209 - precision: 0.8590 - recall: 0.8032


--- Baseline Model Test Results ---
Test Loss: 1.1562
Test Accuracy: 0.6751
Test Precision: 0.6863
Test Recall: 0.6497
Test F1-Score: 0.6675

--- Saving the model to Google Drive ---
Baseline ResNet model saved to /content/drive/MyDrive/MODELS/resnet_baseline.h5
We can now use these results for our report!
